<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/seq2one_return/stage_07_08_transformer_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_08 -  Transformer (many-to-one)**

El Transformer es un modelo basado en self-attention que procesa secuencias completas en paralelo, sin recurrencia. En el esquema many-to-one, el modelo recibe una ventana temporal de múltiples pasos (many) y produce una única salida agregada (one), típicamente usando el embedding del último token o un pooling sobre la secuencia.

Su ventaja clave es capturar dependencias de largo alcance de forma eficiente, con alta escalabilidad y estabilidad en el entrenamiento frente a RNN/LSTM.


# **BLOQUE DE EJECUCIÓN COMPLETO**

# **1. Imports + paths**

In [30]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

In [2]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:

#VENTANAS 60MIN
IN_WINDOW_TRAIN_60_Z = Path(os.environ.get("IN_WINDOW_TRAIN_60_Z", "data/windows_seq2one/seq2one_return/train_delta60_ws60.npz"))
IN_WINDOW_VALID_60_Z = Path(os.environ.get("IN_WINDOW_VALID_60_Z", "data/windows_seq2one/seq2one_return/test_delta60_ws60.npz"))
IN_WINDOW_TEST_60_Z = Path(os.environ.get("IN_WINDOW_TEST_60_Z", "data/windows_seq2one/seq2one_return/valid_delta60_ws60.npz"))

#VENTANAS 90MIN
IN_WINDOW_TRAIN_90_Z = Path(os.environ.get("IN_WINDOW_TRAIN_90_Z", "data/windows_seq2one/seq2one_return/train_delta90_ws60.npz"))
IN_WINDOW_VALID_90_Z = Path(os.environ.get("IN_WINDOW_VALID_90_Z", "data/windows_seq2one/seq2one_return/test_delta90_ws60.npz"))
IN_WINDOW_TEST_90_Z = Path(os.environ.get("IN_WINDOW_TEST_90_Z", "data/windows_seq2one/seq2one_return/valid_delta90_ws60.npz"))

#ESCALADOR GLOBAL
IN_SCALER = Path(os.environ.get("IN_SCALER", "data/scaled/scaler.joblib"))




In [3]:
#ARTIFACTS

# Summary del stage_03a (donde está delta_target_p70 por horizonte).
#IN_TARGET_INVESTIGATION_SUMMARY = Path(os.environ.get("IN_TARGET_INVESTIGATION_SUMMARY", "reports/stage_03a_target_investigation_summary.json"))

# Summary del stage_06 (donde está window_size y n_features por horizonte).
#IN_WINDOWS_SCALING_SUMMARY =Path(os.environ.get("IN_WINDOWS_SCALING_SUMMARY", "reports/stage_06_window_scaling_seq2seq_summary.json"))

OUT_MODEL_METRICS = Path(os.environ.get("OUT_MODEL_METRICS", f"reports/stage_07__model_metrics.json"))

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


In [5]:
#PARA LA NOTEBOOK
IN_WINDOW_TRAIN_60_Z = DRIVE_DIR / IN_WINDOW_TRAIN_60_Z
IN_WINDOW_VALID_60_Z = DRIVE_DIR / IN_WINDOW_VALID_60_Z
IN_WINDOW_TEST_60_Z = DRIVE_DIR / IN_WINDOW_TEST_60_Z

IN_WINDOW_TRAIN_90_Z = DRIVE_DIR / IN_WINDOW_TRAIN_90_Z
IN_WINDOW_VALID_90_Z=DRIVE_DIR / IN_WINDOW_VALID_90_Z
IN_WINDOW_TEST_90_Z = DRIVE_DIR / IN_WINDOW_TEST_90_Z

IN_SCALER = DRIVE_DIR / IN_SCALER

#IN_WINDOWS_SCALING_SUMMARY = DRIVE_DIR / IN_WINDOWS_SCALING_SUMMARY
#IN_TARGET_INVESTIGATION_SUMMARY = DRIVE_DIR / IN_TARGET_INVESTIGATION_SUMMARY

# **2. Reproducibilidad**

In [6]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

# **3. Configuración**

In [7]:
def _read_json(path: Path) -> Dict[str, Any]:
    """Lee un JSON y devuelve un dict Python (con validación básica de existencia)."""
    # Verifica que el archivo exista antes de abrirlo.
    if not path.exists():
        # Si no existe, corta la ejecución con un error claro.
        raise FileNotFoundError(f"No existe el JSON: {path}")
    # Abre el archivo en modo lectura, asegurando UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # Parsea el contenido JSON y lo devuelve como dict.
        return json.load(f)

In [8]:
# Config final para entrenar (modelo, horizonte).
@dataclass(frozen=True)
class StageConfig:
    # Horizonte (60 o 90).
    horizon: int
    # Largo de ventana (timesteps) desde stage_06.
    seq_len: int
    # Cantidad de features desde stage_06 para ese horizonte.
    n_features: int
    # Nombres de features (orden exacto) para ese horizonte.
    feature_names: List[str]
    # Nombre del target para ese horizonte.
    target_name: List[str]
    # Umbral mínimo económico (DELTA_BASE).
    delta_base: float
    # Umbral de oportunidad (DELTA_OP) leído del stage_03a.
    delta_op: float

In [9]:
# Construye un StageConfig leyendo ambos reports.
SUMMARY = """
def load_state_from_reports(
    horizon: int,
    #model_name: str,
    *,
    in_windows_scaling_summary: Path = IN_WINDOWS_SCALING_SUMMARY,
    in_target_investigation_summary: Path = IN_TARGET_INVESTIGATION_SUMMARY,
) -> StageConfig:
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Carga JSON del stage_06.
    w = _read_json(in_windows_scaling_summary)

    # Carga JSON del stage_03a.
    t = _read_json(in_target_investigation_summary)

    # Lee window_size global (SEQ_LEN).
    seq_len = int(w["details"]["config"]["window_size"])

    # Selecciona dataset del horizonte (ojo: "60" o "90" como string).
    ds = w["details"]["datasets"][str(horizon)]

    # Lee n_features del horizonte.
    n_features = int(ds["n_features"])

    # Lee feature_names del horizonte (aquí se refleja el 1 feature distinto).
    feature_names = list(ds["feature_names"])

    # Lee target del horizonte.
    target_name = ds["target"]

    # Valida consistencia.
    if len(feature_names) != n_features:
        raise ValueError("Inconsistencia entre n_features y feature_names")

    # Define la key de delta_base_med del stage_03a.
    target_base = f"h{horizon}_delta_base_med"
    delta_base = float(t["metrics"][target_base])

    # Define la key de delta_target_p70 del stage_03a.
    target_op = f"h{horizon}_delta_target_p70"
    # Lee DELTA_OP para ese horizonte.
    delta_op = float(t["metrics"][target_op])

    # Devuelve la config lista para entrenar.
    return StageConfig(
        horizon=horizon,
        seq_len=seq_len,
        n_features=n_features,
        feature_names=feature_names,
        target_name=target_name,
        delta_base=float(delta_base),
        delta_op=float(delta_op),
            )
"""

In [10]:
#states_h60 = load_state_from_reports(horizon=60)
#states_h60

In [11]:
#states_h90 = load_state_from_reports(horizon=90)
#states_h90

# **4. Importar métricas comunes desde .py**

In [12]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [13]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


# **5. Carga de data windows**

In [14]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [15]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [16]:
# Cargar scaler
#scaler = joblib.load("/content/drive/MyDrive/neural_profit/data/scaled/scaler.joblib")

In [17]:
# --------------------------------------------------
# Carga completa: train/valid/test + scaler por horizonte
# --------------------------------------------------
def load_windows_and_scaler_for_horizon(horizon: int) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler para un horizonte dado (60 o 90).

    Retorna un dict:
    {
      "horizon": 60,
      "paths": {...},
      "scaler": <StandardScaler>,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }
    """
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Selecciona paths según horizonte.
    if horizon == 60:
        train_path = IN_WINDOW_TRAIN_60_Z
        valid_path = IN_WINDOW_VALID_60_Z
        test_path  = IN_WINDOW_TEST_60_Z
        scaler_path = IN_SCALER
    else:
        train_path = IN_WINDOW_TRAIN_90_Z
        valid_path = IN_WINDOW_VALID_90_Z
        test_path  = IN_WINDOW_TEST_90_Z
        scaler_path = IN_SCALER

    # Carga ventanas.
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    # Carga scaler.
    scaler = load_scaler(scaler_path)

    # Retorna todo empaquetado.
    return {
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [18]:
# --------------------------------------------------
# Carga efectiva: H60 y H90 (dos datasets distintos)
# --------------------------------------------------

# Carga todo para 60 min.
bundle_60 = load_windows_and_scaler_for_horizon(60)

# Carga todo para 90 min.
bundle_90 = load_windows_and_scaler_for_horizon(90)


In [19]:
# --------------------------------------------------
# Verificación rápida
# --------------------------------------------------

# Shapes H60.
print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

# Shapes H90.
print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

# Información útil (scaler).
print("Scaler H60:", type(bundle_60["scaler"]).__name__)
print("Scaler H90:", type(bundle_90["scaler"]).__name__)

H60 Train: (330144, 1200) (330144,)
H60 Valid: (70952, 1200) (70952,)
H60 Test : (70590, 1200) (70590,)
H90 Train: (330144, 1200) (330144,)
H90 Valid: (70952, 1200) (70952,)
H90 Test : (70590, 1200) (70590,)
Scaler H60: StandardScaler
Scaler H90: StandardScaler


NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

# **6. Sanity Check**

In [20]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [21]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [22]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [23]:
summary = run_sanity_checks_all_horizons_seq2one(bundle_60, bundle_90)

#summary["h60"]["train"]

[sanity_check_seq2one] train_h60 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=1.008703
[sanity_check_seq2one] valid_h60 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=1.021581
[sanity_check_seq2one] test_h60 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=0.640061
OK h60 (h=60)
[sanity_check_seq2one] train_h90 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=1.008347
[sanity_check_seq2one] valid_h90 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=1.036845
[sanity_check_seq2one] test_h90 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=0.647445
OK h90 (h=90)


# **DEFINICIÓN DE MODELO**

# **7. Definición del modelo — placeholder**

## **7.1. Modelo Transformer (Many to One) — seq2one**

**Idea básica**

El **Transformer** es un modelo de aprendizaje profundo basado en el
mecanismo de **self-attention**, diseñado para modelar dependencias
temporales **sin recurrencia** y con procesamiento completamente
paralelo de la secuencia de entrada.

En el esquema **many-to-one**, el modelo recibe una **ventana temporal**
de múltiples pasos (many) y produce **un único valor escalar futuro**
(one), asociado al final de la ventana.

A diferencia del MLP, el Transformer **preserva explícitamente la
estructura temporal**, permitiendo que cada instante de la secuencia
atienda a cualquier otro instante según su relevancia para la
predicción final.

Formalmente, el modelo puede representarse como:

$$
\hat{y}_t = g\Big( \text{Pool}\big( \text{TransformerEncoder}(X_t) \big) \Big)
$$

donde:
- $X_t \in \mathbb{R}^{T \times F}$ es la ventana temporal (longitud $T$, $F$ features),
- $\text{TransformerEncoder}(\cdot)$ aplica capas de self-attention y feedforward,
- $\text{Pool}(\cdot)$ es una agregación temporal (último token, mean pooling, etc.),
- $g(\cdot)$ es una capa densa final que produce el target escalar.

---

**Regularización (Transformer)**

**Riesgo:** Alto, debido a la elevada capacidad del modelo.

El Transformer incorpora **regularización parcial de forma intrínseca**,
pero requiere control explícito:

- **Dropout (intrínseco):**
  - Aplicado en self-attention y capas feedforward.
- **Early stopping:**
  - Fundamental para evitar sobreajuste.
- **Dimensión del embedding controlada:**
  - Evita representaciones excesivamente complejas.
- **Número limitado de capas encoder:**
  - 1–3 capas en escenarios de datos financieros.
- **Weight decay (opcional):**
  - Refuerza la estabilidad del entrenamiento.

---

**Por qué el Transformer es relevante en este proyecto**

- Capacidad para capturar:
  - dependencias **de largo alcance**,
  - relaciones temporales no locales.
- Adecuado para:
  - ventanas largas (60–90 minutos),
  - múltiples indicadores técnicos simultáneos.
- Entrenamiento:
  - paralelo y estable,
  - más escalable que LSTM/GRU.

El Transformer representa el **primer modelo plenamente atencional**
del pipeline, sirviendo como referencia frente a arquitecturas
secuenciales (LSTM, GRU) y convolucionales (TCN).

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Número de capas encoder: 1–2
- Dimensión del embedding: moderada (32–64)
- Número de cabezas de atención: 2–4
- Dropout: activado
- Pooling temporal: último token o mean pooling
- Early stopping: activado
- **Sin tuning exhaustivo** (optimización posterior)

El ajuste fino de profundidad, atención y regularización se aborda en
etapas posteriores del proyecto.


## **7.2. Imports y “seed” (base reproducible)**

In [24]:
# Paso 1: imports básicos + reproducibilidad (sin tqdm)
import os
import json
import random
from pathlib import Path
from typing import Dict, Any, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [25]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # reproducibilidad (puede bajar performance, pero estable)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


## **7.3. Reconstruir X para H60 y H90 en paralelo**

In [26]:
# Paso 2: reconstruir X a formato secuencial (N, T, F) para Transformer (H60 y H90)

import numpy as np

SEQ_LEN = 60
N_FEATS = 20
FLAT_DIM = SEQ_LEN * N_FEATS

def unflatten_X(X_flat: np.ndarray, *, seq_len: int, n_feats: int) -> np.ndarray:
    assert X_flat.ndim == 2, f"Esperado X_flat (N, D). Recibido: {X_flat.shape}"
    assert X_flat.shape[1] == seq_len * n_feats, (
        f"D={X_flat.shape[1]} no coincide con seq_len*n_feats={seq_len*n_feats}"
    )
    return X_flat.reshape(X_flat.shape[0], seq_len, n_feats)

# --- H60 ---
X_train_60 = unflatten_X(bundle_60["train"]["X"], seq_len=SEQ_LEN, n_feats=N_FEATS)
y_train_60 = bundle_60["train"]["y"]

X_valid_60 = unflatten_X(bundle_60["valid"]["X"], seq_len=SEQ_LEN, n_feats=N_FEATS)
y_valid_60 = bundle_60["valid"]["y"]

X_test_60  = unflatten_X(bundle_60["test"]["X"],  seq_len=SEQ_LEN, n_feats=N_FEATS)
y_test_60  = bundle_60["test"]["y"]

# --- H90 ---
X_train_90 = unflatten_X(bundle_90["train"]["X"], seq_len=SEQ_LEN, n_feats=N_FEATS)
y_train_90 = bundle_90["train"]["y"]

X_valid_90 = unflatten_X(bundle_90["valid"]["X"], seq_len=SEQ_LEN, n_feats=N_FEATS)
y_valid_90 = bundle_90["valid"]["y"]

X_test_90  = unflatten_X(bundle_90["test"]["X"],  seq_len=SEQ_LEN, n_feats=N_FEATS)
y_test_90  = bundle_90["test"]["y"]

print("H60 X_train:", X_train_60.shape, "y_train:", y_train_60.shape)
print("H60 X_valid:", X_valid_60.shape, "y_valid:", y_valid_60.shape)
print("H60 X_test: ", X_test_60.shape,  "y_test: ", y_test_60.shape)

print("H90 X_train:", X_train_90.shape, "y_train:", y_train_90.shape)
print("H90 X_valid:", X_valid_90.shape, "y_valid:", y_valid_90.shape)
print("H90 X_test: ", X_test_90.shape,  "y_test: ", y_test_90.shape)



H60 X_train: (330144, 60, 20) y_train: (330144,)
H60 X_valid: (70952, 60, 20) y_valid: (70952,)
H60 X_test:  (70590, 60, 20) y_test:  (70590,)
H90 X_train: (330144, 60, 20) y_train: (330144,)
H90 X_valid: (70952, 60, 20) y_valid: (70952,)
H90 X_test:  (70590, 60, 20) y_test:  (70590,)


## **7.4. TensorDataset + DataLoader (PyTorch)**

In [27]:
# Paso 3: TensorDataset y DataLoaders para H60 y H90 (sin tqdm)

import torch
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 512   # si falta VRAM: 256 o 128
NUM_WORKERS = 2    # en Colab: 0-2 suele ser estable

def make_loaders(X_train, y_train, X_valid, y_valid, X_test, y_test):
    # Tensores float32
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)

    X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
    y_valid_t = torch.tensor(y_valid, dtype=torch.float32)

    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

    # Datasets
    train_ds = TensorDataset(X_train_t, y_train_t)
    valid_ds = TensorDataset(X_valid_t, y_valid_t)
    test_ds  = TensorDataset(X_test_t,  y_test_t)

    # Loaders
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
    )
    valid_loader = DataLoader(
        valid_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    return train_loader, valid_loader, test_loader

# --- H60 ---
train_loader_60, valid_loader_60, test_loader_60 = make_loaders(
    X_train_60, y_train_60, X_valid_60, y_valid_60, X_test_60, y_test_60
)

# --- H90 ---
train_loader_90, valid_loader_90, test_loader_90 = make_loaders(
    X_train_90, y_train_90, X_valid_90, y_valid_90, X_test_90, y_test_90
)

# Smoke test: un batch por horizonte
xb60, yb60 = next(iter(train_loader_60))
xb90, yb90 = next(iter(train_loader_90))

print("H60 batch X:", xb60.shape, xb60.dtype, "| batch y:", yb60.shape, yb60.dtype)
print("H90 batch X:", xb90.shape, xb90.dtype, "| batch y:", yb90.shape, yb90.dtype)


H60 batch X: torch.Size([512, 60, 20]) torch.float32 | batch y: torch.Size([512]) torch.float32
H90 batch X: torch.Size([512, 60, 20]) torch.float32 | batch y: torch.Size([512]) torch.float32


## **7.5. Definición de modelo Transformer (many-to-one)**

Esta implementación usa:
- Proyección: 20 → d_model
- Positional encoding (sin metadata)
- TransformerEncoder
- Pooling: mean sobre el tiempo
- Head: salida escalar

In [31]:
# ============================================================
# Paso 4: Modelo Transformer many-to-one (regresión)
# Compatible con H=60 y H=90 (modelos independientes)
# ============================================================

import math
import torch
import torch.nn as nn


# ------------------------------------------------------------
# Positional Encoding (sinusoidal)
# ------------------------------------------------------------
class PositionalEncoding(nn.Module):
    """
    Inyecta información temporal explícita en la secuencia.
    Implementación sinusoidal estándar (Vaswani et al.).
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=float(dropout))

        d_model = int(d_model)
        max_len = int(max_len)

        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, T, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        T = x.size(1)
        x = x + self.pe[:, :T, :]
        return self.dropout(x)


# ------------------------------------------------------------
# Transformer Many-to-One (Regresión)
# ------------------------------------------------------------
class TransformerManyToOne(nn.Module):
    """
    Entrada : (B, T, F)
    Salida  : (B,)
    """
    def __init__(
        self,
        n_features: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_ff: int = 128,
        dropout: float = 0.1,
        pooling: str = "mean",  # "mean" | "last"
        max_len: int = 1000,
    ):
        super().__init__()

        n_features = int(n_features)
        d_model = int(d_model)
        nhead = int(nhead)
        num_layers = int(num_layers)
        dim_ff = int(dim_ff)
        dropout = float(dropout)
        max_len = int(max_len)

        assert d_model % nhead == 0, "d_model debe ser divisible por nhead"
        assert pooling in ("mean", "last"), "pooling inválido (use 'mean' o 'last')"
        self.pooling = pooling

        # 1) Proyección inicial: features -> embedding
        self.input_proj = nn.Linear(n_features, d_model)

        # 2) Positional encoding
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout, max_len=max_len)

        # 3) Transformer encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,     # (B, T, D)
            activation="gelu",
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # 4) Head de regresión
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        z = self.input_proj(x)     # (B, T, D)
        z = self.pos_enc(z)        # (B, T, D)
        z = self.encoder(z)        # (B, T, D)

        if self.pooling == "mean":
            z_pool = z.mean(dim=1)     # (B, D)
        else:
            z_pool = z[:, -1, :]       # (B, D)

        y_hat = self.head(z_pool).squeeze(-1)  # (B,)
        return y_hat


# ------------------------------------------------------------
# Ejemplo de uso (H=60 / H=90: modelos independientes)
# ------------------------------------------------------------
# device = "cuda" if torch.cuda.is_available() else "cpu"

# model_60 = TransformerManyToOne(
#     n_features=20, d_model=64, nhead=4, num_layers=2, dim_ff=128, dropout=0.1, pooling="mean"
# ).to(device)

# model_90 = TransformerManyToOne(
#     n_features=20, d_model=64, nhead=4, num_layers=2, dim_ff=128, dropout=0.1, pooling="mean"
# ).to(device)

# xb60, yb60 = next(iter(train_loader_60))
# xb60 = xb60.to(device)
# with torch.no_grad():
#     out60 = model_60(xb60)
# print("model_60 out:", out60.shape, out60.dtype)


In [32]:
# ------------------------------------------------------------
# Instanciación de DOS modelos (H60 y H90) - independientes
# ------------------------------------------------------------
model_60 = TransformerManyToOne(
    n_features=20,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_ff=128,
    dropout=0.1,
    pooling="mean",
).to(device)

model_90 = TransformerManyToOne(
    n_features=20,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_ff=128,
    dropout=0.1,
    pooling="mean",
).to(device)

# ------------------------------------------------------------
# Smoke test (uno por horizonte)
# ------------------------------------------------------------
xb60, yb60 = next(iter(train_loader_60))
xb90, yb90 = next(iter(train_loader_90))

xb60 = xb60.to(device)
xb90 = xb90.to(device)

with torch.no_grad():
    out60 = model_60(xb60)
    out90 = model_90(xb90)

print("model_60 out:", out60.shape, out60.dtype)
print("model_90 out:", out90.shape, out90.dtype)


model_60 out: torch.Size([512]) torch.float32
model_90 out: torch.Size([512]) torch.float32


## **7.6. Definición de loss, optimizer y funciones de train / eval (sin tqdm)**

In [33]:
#loss, optimizer y funciones de entrenamiento / evaluación

import torch
import torch.nn as nn
from typing import Dict


# ------------------------------------------------------------
# Loss (regresión)
# ------------------------------------------------------------
criterion = nn.MSELoss()


# ------------------------------------------------------------
# Optimizers (uno por modelo, independientes)
# ------------------------------------------------------------
optimizer_60 = torch.optim.AdamW(
    model_60.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

optimizer_90 = torch.optim.AdamW(
    model_90.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)



# ------------------------------------------------------------
# Función de entrenamiento (1 epoch)
# ------------------------------------------------------------
def train_one_epoch(
    model: nn.Module,
    loader,
    optimizer,
    criterion,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        y_hat = model(xb)
        loss = criterion(y_hat, yb)

        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / n_samples


# ------------------------------------------------------------
# Función de evaluación (1 epoch)
# ------------------------------------------------------------
@torch.no_grad()
def eval_one_epoch(
    model: nn.Module,
    loader,
    criterion,
    device: torch.device,
) -> float:
    model.eval()
    total_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        y_hat = model(xb)
        loss = criterion(y_hat, yb)

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / n_samples


## **7.7. Loop de entrenamiento completo con early stopping**



In [34]:
# Paso 6.1: configuración de entrenamiento (común)

import math

MAX_EPOCHS = 30
PATIENCE = 5
MIN_DELTA = 0.0
SAVE_BEST = True

In [35]:
# Paso 6.2: entrenamiento Transformer H60 (early stopping)

BEST_PATH_60 = "best_transformer_60.pt"

best_val_60 = math.inf
best_epoch_60 = -1
patience_left_60 = PATIENCE

history_60 = {
    "train_loss": [],
    "valid_loss": [],
}

best_state_60 = None

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(
        model_60, train_loader_60, optimizer_60, criterion, device
    )
    val_loss = eval_one_epoch(
        model_60, valid_loader_60, criterion, device
    )

    history_60["train_loss"].append(float(train_loss))
    history_60["valid_loss"].append(float(val_loss))

    improved = (best_val_60 - val_loss) > MIN_DELTA
    if improved:
        best_val_60 = val_loss
        best_epoch_60 = epoch
        patience_left_60 = PATIENCE

        if SAVE_BEST:
            best_state_60 = {
                k: v.detach().cpu().clone()
                for k, v in model_60.state_dict().items()
            }
            torch.save(model_60.state_dict(), BEST_PATH_60)
    else:
        patience_left_60 -= 1

    print(
        f"[H60] epoch {epoch:02d} | "
        f"train_loss={train_loss:.6f} | "
        f"valid_loss={val_loss:.6f} | "
        f"patience_left={patience_left_60}"
    )

    if patience_left_60 <= 0:
        print(
            f"[H60] Early stopping: "
            f"best_valid_loss={best_val_60:.6f} "
            f"at epoch {best_epoch_60}"
        )
        break

if SAVE_BEST and best_state_60 is not None:
    model_60.load_state_dict(best_state_60)
    model_60.to(device)
    print(
        f"[H60] Modelo restaurado: "
        f"epoch {best_epoch_60} | "
        f"best_valid_loss={best_val_60:.6f}"
    )

[H60] epoch 01 | train_loss=1.003675 | valid_loss=1.045930 | patience_left=5
[H60] epoch 02 | train_loss=0.961602 | valid_loss=1.126412 | patience_left=4
[H60] epoch 03 | train_loss=0.919659 | valid_loss=1.134340 | patience_left=3
[H60] epoch 04 | train_loss=0.878874 | valid_loss=1.158396 | patience_left=2
[H60] epoch 05 | train_loss=0.844182 | valid_loss=1.111976 | patience_left=1
[H60] epoch 06 | train_loss=0.805013 | valid_loss=1.279299 | patience_left=0
[H60] Early stopping: best_valid_loss=1.045930 at epoch 1
[H60] Modelo restaurado: epoch 1 | best_valid_loss=1.045930


In [36]:
# Paso 6.3: entrenamiento Transformer H90 (early stopping)

BEST_PATH_90 = "best_transformer_90.pt"

best_val_90 = math.inf
best_epoch_90 = -1
patience_left_90 = PATIENCE

history_90 = {
    "train_loss": [],
    "valid_loss": [],
}

best_state_90 = None

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(
        model_90, train_loader_90, optimizer_90, criterion, device
    )
    val_loss = eval_one_epoch(
        model_90, valid_loader_90, criterion, device
    )

    history_90["train_loss"].append(float(train_loss))
    history_90["valid_loss"].append(float(val_loss))

    improved = (best_val_90 - val_loss) > MIN_DELTA
    if improved:
        best_val_90 = val_loss
        best_epoch_90 = epoch
        patience_left_90 = PATIENCE

        if SAVE_BEST:
            best_state_90 = {
                k: v.detach().cpu().clone()
                for k, v in model_90.state_dict().items()
            }
            torch.save(model_90.state_dict(), BEST_PATH_90)
    else:
        patience_left_90 -= 1

    print(
        f"[H90] epoch {epoch:02d} | "
        f"train_loss={train_loss:.6f} | "
        f"valid_loss={val_loss:.6f} | "
        f"patience_left={patience_left_90}"
    )

    if patience_left_90 <= 0:
        print(
            f"[H90] Early stopping: "
            f"best_valid_loss={best_val_90:.6f} "
            f"at epoch {best_epoch_90}"
        )
        break

if SAVE_BEST and best_state_90 is not None:
    model_90.load_state_dict(best_state_90)
    model_90.to(device)
    print(
        f"[H90] Modelo restaurado: "
        f"epoch {best_epoch_90} | "
        f"best_valid_loss={best_val_90:.6f}"
    )


[H90] epoch 01 | train_loss=0.999137 | valid_loss=1.090546 | patience_left=5
[H90] epoch 02 | train_loss=0.954973 | valid_loss=1.098149 | patience_left=4
[H90] epoch 03 | train_loss=0.913171 | valid_loss=1.149128 | patience_left=3
[H90] epoch 04 | train_loss=0.872381 | valid_loss=1.116649 | patience_left=2
[H90] epoch 05 | train_loss=0.835973 | valid_loss=1.228545 | patience_left=1
[H90] epoch 06 | train_loss=0.804784 | valid_loss=1.178364 | patience_left=0
[H90] Early stopping: best_valid_loss=1.090546 at epoch 1
[H90] Modelo restaurado: epoch 1 | best_valid_loss=1.090546


## **7.6. Predicciones Transformer**


Función de predicción (seq2one) -> y_true, y_pred

In [37]:
import numpy as np
import torch

@torch.no_grad()
def predict_seq2one(
    model,
    loader,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()

    y_true_list = []
    y_pred_list = []

    for xb, yb in loader:
        xb = xb.to(device)

        y_hat = model(xb).detach().cpu().numpy()
        y_true = yb.detach().cpu().numpy()

        y_pred_list.append(y_hat)
        y_true_list.append(y_true)

    y_true_all = np.concatenate(y_true_list, axis=0)
    y_pred_all = np.concatenate(y_pred_list, axis=0)

    return y_true_all, y_pred_all


In [38]:
y_true_valid_60, y_pred_valid_60 = predict_seq2one(model_60, valid_loader_60, device)
y_true_test_60,  y_pred_test_60  = predict_seq2one(model_60, test_loader_60,  device)

print("H60 valid:", y_true_valid_60.shape, y_pred_valid_60.shape)
print("H60 test :", y_true_test_60.shape,  y_pred_test_60.shape)

H60 valid: (70952,) (70952,)
H60 test : (70590,) (70590,)


In [39]:
y_true_valid_90, y_pred_valid_90 = predict_seq2one(model_90, valid_loader_90, device)
y_true_test_90,  y_pred_test_90  = predict_seq2one(model_90, test_loader_90,  device)

print("H90 valid:", y_true_valid_90.shape, y_pred_valid_90.shape)
print("H90 test :", y_true_test_90.shape,  y_pred_test_90.shape)

H90 valid: (70952,) (70952,)
H90 test : (70590,) (70590,)


# **8. Métricas ML**

In [40]:
import pandas as pd

def metrics_to_df(metrics: dict, *, model: str, split: str, horizon: int) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """
    return pd.DataFrame([{
        "model": model,
        "split": split,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])


In [41]:
import pandas as pd

# ============================================================
# H=60
# ============================================================
metrics_valid_60 = compute_seq2one_metrics(y_true_valid_60, y_pred_valid_60, compute_r2=True)
metrics_test_60  = compute_seq2one_metrics(y_true_test_60,  y_pred_test_60,  compute_r2=True)

df_valid_60 = metrics_to_df(metrics_valid_60, model="transformer", split="valid", horizon=60)
df_test_60  = metrics_to_df(metrics_test_60,  model="transformer", split="test",  horizon=60)

# ============================================================
# H=90
# ============================================================
metrics_valid_90 = compute_seq2one_metrics(y_true_valid_90, y_pred_valid_90, compute_r2=True)
metrics_test_90  = compute_seq2one_metrics(y_true_test_90,  y_pred_test_90,  compute_r2=True)

df_valid_90 = metrics_to_df(metrics_valid_90, model="transformer", split="valid", horizon=90)
df_test_90  = metrics_to_df(metrics_test_90,  model="transformer", split="test",  horizon=90)

# ============================================================
# Tabla final (VALID+TEST, H60+H90)
# ============================================================
df_transformer_metrics = pd.concat(
    [df_valid_60, df_valid_90, df_test_60, df_test_90],
    ignore_index=True
).sort_values(["split", "horizon_min"]).reset_index(drop=True)

df_transformer_metrics


,model,split,horizon_min,MAE,RMSE,R2,DA
0,transformer,test,60,0.472632,0.640124,-0.000197,0.508514
1,transformer,test,90,0.480926,0.653129,-0.017635,0.477830
2,transformer,valid,60,0.649394,1.022707,-0.002205,0.499605
3,transformer,valid,90,0.665879,1.044292,-0.014416,0.491431


# **9. Guardar artefactos para Stage_08**

In [42]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_return",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"metrics_{name}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path


In [43]:
save_seq2one_metrics(
    df_transformer_metrics,
    name="transformer",
)

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_return/metrics_transformer.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_return/metrics_transformer.parquet')